# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/12-kartik66/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

**Lane: Refresh / Content Opportunity Scoring**  
This notebook maps the lane onto the ML loop — naming the task type,
target, metric, unit of analysis, and why learning beats a fixed rule.

## 1. My lane as an ML task (type)

**Task type: Scoring (priority scoring, which drives a ranking).**  

This is **not** classification (we do not need a yes/no gate — we need
an ordered queue). It is **scoring**: every page gets a continuous
priority score, and the reviewer works from the top down. The output
is a ranked list answering: *"Which pages should be reviewed first?"*

This maps to the `ranking / scoring` row in the skill table —
the question is "which ones first?", the target is a priority score,
and the metric is precision@K.

**Decision it improves:** A content editor deciding which 20 pages to
review today, out of thousands. Without scoring, they pick by
recency or gut feel. With scoring, they pick by evidence.

**Who acts on it:** A content operations reviewer. They open the ranked
queue, inspect the top-K pages, and decide which to refresh, rewrite,
or monitor. Each page carries a reason code so the review is not blind.

In [1]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Confirm grain: one row per content item
print('Rows (content items):', len(df))
print('Unique content_ids:', df['content_id'].nunique())
print('Unique clients:', df['client_id'].nunique())
print('Columns:', df.shape[1])
print()
print('Grain check (duplicate content_ids):', df['content_id'].duplicated().sum())

Rows (content items): 30000
Unique content_ids: 30000
Unique clients: 32
Columns: 44

Grain check (duplicate content_ids): 0


## 2. Target or proxy

**Target: `is_declining_label` — a binary flag: is this page currently
in decline?**  

The label is defined as:  
`is_declining_label = 1` when `trend_direction == "down"`, else 0.
`trend_direction` compares `impressions_last_30d` vs
`impressions_prev_30d`: if impressions dropped more than 20%,
the direction is `"down"`.

**This is a proxy.** The ideal target would be a *future* outcome —
"will this page decline in the next 30 days?" — measured from a
forward time window in the warehouse daily data. What we have in
the starter dataset is a *current-state* label: it flags pages that
are already declining now. This is useful for prioritising review
attention, but it is a proxy for the thing we really want to
predict (future risk).

**Where does the label come from?** The `trend_direction` column is
derived from `trend_pct`, which compares two trailing 30-day windows
of impressions. Both are **observed measurements** (actual GSC
impression counts), but the `"down"` threshold (>20% drop) is a
**defined rule**. The label is therefore a rule-applied-to-observed-data
proxy — acceptable for a starter model, but honest about its limits.

**Critical: never leak.** `trend_direction` and `trend_pct` must never
be features — they directly encode the label.

In [2]:
# Create the label column
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print('Label distribution:')
print(df['is_declining_label'].value_counts().to_string())
print()
print('Decline rate: %.2f%%' % (df['is_declining_label'].mean() * 100))
print()
print('Rows where is_declining_label = 1:', (df['is_declining_label'] == 1).sum())
print('Rows where is_declining_label = 0:', (df['is_declining_label'] == 0).sum())
print()
print('How many trend_direction values map to each label:')
print(df.groupby('trend_direction')['is_declining_label'].first().to_string())

Label distribution:
is_declining_label
1    16262
0    13738

Decline rate: 54.21%

Rows where is_declining_label = 1: 16262
Rows where is_declining_label = 0: 13738

How many trend_direction values map to each label:
trend_direction
down      1
flat      0
new       0
stable    0
up        0


## 3. Success metric

**Primary metric: Precision@K (specifically Precision@20 or @50).**  

Why: The real decision is a review queue. A content reviewer checks
a fixed number of pages per day — say 20. Precision@K asks: *of the
top K pages the system flags, how many are actually declining?*
This matches how the output is used.

**Secondary metric: Average Precision (AP).** AP summarises precision
across all recall levels — it rewards systems that rank true positives
early. This is the single-number summary for model selection.

**Backup metric: ROC-AUC,** for comparing against the starter model
results in the lane guide (baseline 0.627, RF 0.750 on this slice).
ROC-AUC is useful for model comparison but does not reflect queue
usage directly.

**What number means "good"?** On this starter slice, the lane guide
reports baseline Precision@50 = 0.240 (12/50 correct) and
Random Forest Precision@50 = 0.740 (37/50 correct). A good model
should beat 0.240 — ideally landing above 0.600 on Precision@50
in cross-validation. These numbers describe the starter slice only;
the full warehouse will demand new numbers earned honestly.

In [3]:
# Show base rate and what random precision looks like
base_rate = df['is_declining_label'].mean()
print('Base rate (overall decline proportion): %.3f' % base_rate)
print('Random precision@50 would be: %.3f (%.0f/50)' % (base_rate, base_rate * 50))
print()
print('Best possible Precision@50 given base rate: %.3f' % min(1.0, 50/df['is_declining_label'].sum()))
print('(limited by number of true positives in the dataset)')
print()
print('Lane guide benchmark (starter slice, Precision@50):')
print('  Baseline rules:      0.240 (12/50)')
print('  Logistic regression: 0.400 (20/50)')
print('  Decision tree:       0.540 (27/50)')
print('  Random forest:       0.740 (37/50)')

Base rate (overall decline proportion): 0.542
Random precision@50 would be: 0.542 (27/50)

Best possible Precision@50 given base rate: 0.003
(limited by number of true positives in the dataset)

Lane guide benchmark (starter slice, Precision@50):
  Baseline rules:      0.240 (12/50)
  Logistic regression: 0.400 (20/50)
  Decision tree:       0.540 (27/50)
  Random forest:       0.740 (37/50)


## 4. The unit of analysis, as a real dataframe

**One row = one content item (one web page/article).**  

Each row describes a page's search performance over the trailing
90 days: impressions, clicks, sessions, engagement, position, age,
keyword context, and the current trend direction.

The dataframe below shows the actual data loaded from
`data/raw/content_refresh_anonymized.csv`. I select a representative
set of columns to show the unit of analysis clearly: identifiers,
content properties, traffic metrics, and the label.

In [4]:
# Show the unit of analysis as a real dataframe
cols_to_show = [
    'content_id', 'client_id', 'content_type', 'content_age_days',
    'impressions_90d', 'clicks_90d', 'sessions_90d',
    'ctr', 'avg_position', 'engagement_rate',
    'trend_direction', 'is_declining_label'
]

display_df = df[cols_to_show].copy()
print('Unit of analysis: one row = one content item (web page)')
print('Shape:', display_df.shape)
print()
display_df.head(10)

Unit of analysis: one row = one content item (web page)
Shape: (30000, 12)



,content_id,client_id,content_type,content_age_days,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,3803,29,17,0.76,10.6,5.88,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,15320,7,9,0.05,20.3,0.00,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,12581,11,11,0.09,36.5,0.00,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,11751,58,78,0.49,6.2,1.28,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,19140,24,145,0.13,44.0,0.00,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,3970,1,5,0.03,8.5,0.00,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,0,1,0.00,7.0,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,1724,1,28,0.06,21.2,3.57,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,32574,29,68,0.09,46.0,5.88,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,1240,2,3,0.16,4.9,0.00,down,1


## 5. Why ML beats a fixed rule here

**A fixed rule is not enough because the pattern is multivariate,
non-linear, and signal interactions matter.**  

Consider the existing rule: `trend_direction == "down"` when
impressions dropped >20%. This is a **univariate threshold** — it
looks at one dimension (impression change) and ignores everything
else: content type, age, position, engagement, keyword competition,
etc. A page with -30% impressions but low volume (50 impressions)
is flagged the same as one with -30% on 50,000 impressions — but
their review priority should differ enormously.

**Three specific reasons learning beats a rule:**

1. **Non-linear effects.** A page at position 2 with 0.5% CTR may be
   fine. The same CTR at position 8 is terrible. A rule would need
   dozens of nested if-statements to capture position-vs-CTR curves.
   A model learns this from data.

2. **Interaction between signals.** High impressions + low CTR suggests
   a metadata problem. Low impressions + low engagement suggests a
   topic-demand problem. Same individual values, very different
   situations — a rule can't adapt per combination without exploding
   in complexity.

3. **Continuous priority, not binary flags.** A rule gives a yes/no
   ("declining or not"). An editor needs a *ranked queue* — which
   page is worst first? A scoring model produces a continuous
   priority score, ordering all pages by evidence strength.

**What ML adds:** It learns weights and interactions from observed
data, producing a score that ranks pages by evidence of decline
risk. The lane guide's own numbers confirm this: baseline rule
Precision@50 = 0.240 versus Random Forest = 0.740 on the starter
slice.

In [5]:
# Demonstrate: the fixed rule only sees one thing (trend direction)
# Show that pages with the same trend_direction have very different profiles
declining = df[df['trend_direction'] == 'down'].copy()

print('Pages flagged "down" by the fixed rule (n = %d):' % len(declining))
print()
print('Impression range:    %d to %d' % (declining['impressions_90d'].min(), declining['impressions_90d'].max()))
print('CTR range:           %.2f%% to %.2f%%' % (declining['ctr'].min(), declining['ctr'].max()))
print('Position range:      %.1f to %.1f' % (declining['avg_position'].min(), declining['avg_position'].max()))
print('Age range:           %d to %d days' % (declining['content_age_days'].min(), declining['content_age_days'].max()))
print()
print('A rule treats all these pages identically. A model scores them by priority.')
print()

# Show a few contrasting examples
cols = ['content_id', 'impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'content_type', 'is_declining_label']
examples = pd.concat([
    declining.nlargest(3, 'impressions_90d')[cols],
    declining.nsmallest(3, 'impressions_90d')[cols]
])
print('Contrast: high-volume vs low-volume pages, all flagged "down":')
examples

Pages flagged "down" by the fixed rule (n = 16262):

Impression range:    1 to 517715
CTR range:           0.00% to 100.00%
Position range:      0.0 to 118.0
Age range:           90 to 557 days

A rule treats all these pages identically. A model scores them by priority.

Contrast: high-volume vs low-volume pages, all flagged "down":


,content_id,impressions_90d,ctr,avg_position,content_age_days,content_type,is_declining_label
6653,content_5fe46e04994d,517715,0.14,4.2,537,keyword article,1
26844,content_8c19996aa890,509252,0.15,2.5,445,keyword article,1
21819,content_4c36c775b818,463103,0.41,2.3,445,keyword article,1
1162,content_79fbccf1a0d3,1,0.00,1.0,228,keyword article,1
1725,content_8c482a64a3df,1,0.00,3.0,223,keyword article,1
2951,content_0fb0b64ab3fa,1,0.00,5.0,125,keyword article,1


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

---
**Framing summary:**  
For content reviewers deciding *which pages to review first*, we will build a
**priority score** from content and search-engagement signals, predicting
**current decline (is_declining_label proxy)** measured by **Precision@K**.
A wrong call costs an editor's limited review time on the wrong page.
A plain rule is not enough because decline patterns are multivariate,
non-linear, and signal-dependent — requiring learned weights, not
fixed thresholds. We will claim only **observed, decision-support** results.